In [ ]:
#| default_exp augmentations

# Augmentations

> Because this will help

In [ ]:
#| hide
from nbdev.showdoc import *
from torch.utils.data import Dataset, DataLoader

In [ ]:
#| export
import torch, torch.nn.functional as F
from lightning.pytorch.callbacks import Callback
from torch.distributions.beta import Beta
from torchvision.transforms import RandomCrop


## Patching

In [ ]:
#| export 
def create_patch(xb, patch_len, stride, constant_pad=False, constant_pad_value=0):
    """
    xb: [bs x n_vars x seq_len]
    out: [bs x num_patch x n_vars x patch_len]
    """
    if xb.dim() == 2:
        xb = xb.unsqueeze(0)
    if xb.is_nested:
        x_list = list(xb.unbind())
        seq_lens = [x.shape[-1] for x in x_list]
        for i, seq_len in enumerate(seq_lens):
            if constant_pad:
                if ((seq_len-patch_len) % stride != 0):
                    x_list[i] = F.pad(x_list[i], (0, stride), 'constant', value=constant_pad_value)
            x_list[i] = x_list[i].permute(1,0)
            x_list[i] = x_list[i].unfold(dimension=0, size=patch_len, step=stride)
        xb = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
    else:
        if constant_pad:
            seq_len = xb.shape[-1]
            # seq_len > patch_len and stride <= seq_len - patch_len
            if ((seq_len-patch_len) % stride != 0):
                # only pad if remainder
                xb = F.pad(xb, (0, stride), 'constant', value=constant_pad_value) # pad at the end with value
        xb = xb.permute(0,2,1) # xb: [bs x tgt_len x nvars]
        xb = xb.unfold(dimension=1, size=patch_len, step=stride) # xb: [bs x num_patch x n_vars x patch_len]
    return xb

def create_patch3d(xb, frame_len, frame_stride, patch_len, patch_stride, constant_pad=False, constant_pad_value=0):
    """
    xb: [bs x n_vars x seq_len]
    """
    if xb.dim() == 2:
        xb = xb.unsqueeze(0)
    if xb.is_nested:
        x_list = list(xb.unbind())
        seq_lens = [x.shape[-1] for x in x_list]
        for i, seq_len in enumerate(seq_lens):
            if constant_pad:
                if ((seq_len-frame_len) % frame_stride != 0):
                    x_list[i] = F.pad(x_list[i], (0, frame_stride), 'constant', value=constant_pad_value)
            x_list[i] = x_list[i].unfold(dimension=1, size=frame_len, step=frame_stride) # [n_channels, n_frames, frame_len]
            n_channels, n_frames, frame_len = x_list[i].shape
            x_list[i] = x_list[i].unfold(dimension=2, size=patch_len, step=patch_stride)

        xb = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
    else:
        if constant_pad:
            seq_len = xb.shape[-1]
            # seq_len > patch_len and stride <= seq_len - patch_len
            if ((seq_len-frame_len) % frame_stride != 0):
                # only pad if remainder
                xb = F.pad(xb, (0, frame_stride), 'constant', value=constant_pad_value) # pad at the end with value
        xb = xb.unfold(dimension=2, size=frame_len, step=frame_stride)  # [bs x n_channels x n_frames x frame_len]
        # Reshape to prepare for second unfolding
        bs, n_channels, n_frames, frame_len = xb.shape
        #xb = xb.reshape(bs * n_frames, n_channels, frame_len)
        # Second unfold for patches within each frame
        xb = xb.unfold(dimension=2, size=patch_len, step=patch_len)  # [bs*n_frames x n_channels x n_patches_frame x patch_len]
        # Reshape back to 5D
        xb = xb.reshape(bs, n_frames, n_channels, -1, patch_len)  # [bs x n_frames x n_channels x n_patches_frame x patch_len]
        # Permute to get desired dimension order if needed
        xb = xb.permute(0, 2, 1, 3, 4)  # [bs x n_channels x n_frames x n_patches_frame x patch_len]
    return xb
    


In [ ]:
#| notest
batch_size = 2
n_vars = 256
max_len = 100

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

# test seq_len > patch len == stride 
xb = create_patch(x_nested, patch_len=10, stride=5, constant_pad=False)
#xb_rep = create_patch(x_nested, patch_len=500, stride=500, constant_pad=True)
#x_nested.shape, xb.shape, xb_rep.shape
#xb_rep_short = create_patch(x_short, patch_en=502, stride=500, replication_pad=False)b

In [ ]:
#| notest
batch_size = 2
n_vars = 7
max_len = 100

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

# test seq_len > patch len == stride 
xb = create_patch3d(x_nested, patch_len=4, patch_stride=3, frame_len=9, frame_stride=9, constant_pad=True)
#xb_rep = create_patch(x_nested, patch_len=500, stride=500, constant_pad=True)
#x_nested.shape, xb.shape, xb_rep.shape
#xb_rep_short = create_patch(x_short, patch_en=502, stride=500, replication_pad=False)b

In [ ]:
#| notest
x = torch.randn(1, 7, 100)
xb = create_patch(x, patch_len=10, stride=10, constant_pad=True)
xb.shape


torch.Size([1, 10, 7, 10])

In [ ]:
#| export 
def create_patch3d(xb, frame_len, frame_stride, patch_len, patch_stride, constant_pad=False, constant_pad_value=0):
    """
    xb: [bs x n_vars x seq_len]
    """
    if xb.dim() == 2:
        xb = xb.unsqueeze(0)
    if xb.is_nested:
        x_list = list(xb.unbind())
        seq_lens = [x.shape[-1] for x in x_list]
        for i, seq_len in enumerate(seq_lens):
            if constant_pad:
                if ((seq_len-frame_len) % frame_stride != 0):
                    x_list[i] = F.pad(x_list[i], (0, frame_stride), 'constant', value=constant_pad_value)
            x_list[i] = x_list[i].unfold(dimension=1, size=frame_len, step=frame_stride) # [n_channels, n_frames, frame_len]
            n_channels, n_frames, frame_len = x_list[i].shape
            x_list[i] = x_list[i].unfold(dimension=2, size=patch_len, step=patch_stride)

        xb = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
    else:
        if constant_pad:
            seq_len = xb.shape[-1]
            # seq_len > patch_len and stride <= seq_len - patch_len
            if ((seq_len-frame_len) % frame_stride != 0):
                # only pad if remainder
                xb = F.pad(xb, (0, frame_stride), 'constant', value=constant_pad_value) # pad at the end with value
        xb = xb.unfold(dimension=2, size=frame_len, step=frame_stride)  # [bs x n_channels x n_frames x frame_len]
        # Reshape to prepare for second unfolding
        bs, n_channels, n_frames, frame_len = xb.shape
        #xb = xb.reshape(bs * n_frames, n_channels, frame_len)
        # Second unfold for patches within each frame
        xb = xb.unfold(dimension=2, size=patch_len, step=patch_len)  # [bs*n_frames x n_channels x n_patches_frame x patch_len]
        # Reshape back to 5D
        xb = xb.reshape(bs, n_frames, n_channels, -1, patch_len)  # [bs x n_frames x n_channels x n_patches_frame x patch_len]
        # Permute to get desired dimension order if needed
        xb = xb.permute(0, 2, 1, 3, 4)  # [bs x n_channels x n_frames x n_patches_frame x patch_len]
    return xb
    


In [ ]:
#| export
def unpatch(x, seq_len, remove_padding=True):
    """
    x: [bs/None x patch_num x n_vars x patch_len]
    returns x: [bs x n_vars x seq_len]
    """
    if x.dim() == 3:
        x = x.transpose(0,1)
    else:
        x = x.transpose(1,2)
    if x.is_nested:
        x_list = list(x.unbind())
        for i in range(len(x_list)):
            x_list[i] = x_list[i].flatten(start_dim=1)
            if remove_padding:
                x_list[i] = x_list[i][..., :seq_len[i]]
        x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
    else:
        x = x.flatten(start_dim=-2, end_dim=-1)
        if remove_padding:
            x = x[...,:seq_len]
    return x

In [ ]:
#| notest
x = torch.randn(1,1,50)

# test seq_len > patch len == stride 
xb = create_patch(x, patch_len=6, stride=6, constant_pad=True)
xb = unpatch(xb, seq_len=50, remove_padding=False)
xb.shape

torch.Size([1, 1, 54])

## Patch Masking

In [ ]:
#| export
def random_masking(xb, mask_ratio):
    # xb: [bs x num_patch x n_vars x patch_len]
    bs, L, nvars, D = xb.shape
    x = xb.clone()
    
    len_keep = int(L * (1 - mask_ratio))
        
    noise = torch.rand(bs, L, nvars,device=xb.device)  # noise in [0, 1], bs x L x nvars
        
    # sort noise for each sample
    ids_shuffle = torch.argsort(noise, dim=1)  # ascend: small is keep, large is remove
    ids_restore = torch.argsort(ids_shuffle, dim=1)                                     # ids_restore: [bs x L x nvars]

    # keep the first subset
    ids_keep = ids_shuffle[:, :len_keep, :]                                              # ids_keep: [bs x len_keep x nvars]         
    x_kept = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, 1, D))     # x_kept: [bs x len_keep x nvars  x patch_len]
   
    # removed x
    x_removed = torch.zeros(bs, L-len_keep, nvars, D, device=xb.device)                 # x_removed: [bs x (L-len_keep) x nvars x patch_len]
    x_ = torch.cat([x_kept, x_removed], dim=1)                                          # x_: [bs x L x nvars x patch_len]

    # combine the kept part and the removed one
    x_masked = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1,1,1,D)) # x_masked: [bs x num_patch x nvars x patch_len]

    # generate the binary mask: 0 is keep, 1 is remove
    mask = torch.ones([bs, L, nvars], device=x.device)                                  # mask: [bs x num_patch x nvars]
    mask[:, :len_keep, :] = 0
    # unshuffle to get the binary mask
    mask = torch.gather(mask, dim=1, index=ids_restore)                                  # [bs x num_patch x nvars]
    return x_masked, mask

def mask_patches_simple(xb, mask_ratio):
    """
    Function that masks patches using fixed ratio approach similar to random_masking
    
    xb: [bs x patch_num x n_vars x patch_len] or nested tensor
    Returns:
        x_masked: masked tensor with same shape as input
        mask: binary mask where 1 indicates masked positions (bs x patch_num x n_vars)
    """
    if xb.dim() == 3:
        xb = xb.unsqueeze(0)
    x = xb.clone()
    
    if x.is_nested:
        x_list = list(x.unbind())
        for i, x_i in enumerate(x_list):
            # Get dimensions for this tensor in the nested set
            L, nvars, D = x_i.shape
            len_keep = int(L * (1 - mask_ratio))
            
            # Generate random noise for selecting which patches to keep/mask
            noise = torch.rand(L, nvars, device=xb.device)
            # Sort noise to determine keep/mask ordering (smallest values are kept)
            ids_shuffle = torch.argsort(noise, dim=0)  # sort noise for each variable
            # Create restore indices to return to original order later
            ids_restore = torch.argsort(ids_shuffle, dim=0)
            
            # Select patches to keep (those with smallest noise values)
            ids_keep = ids_shuffle[:len_keep]
            # Gather kept patches and their corresponding values
            x_kept = torch.gather(x_i, dim=0, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))
            
            # Create and combine with zeros for masked positions
            # Create tensor of zeros for masked positions
            x_removed = torch.zeros(L - len_keep, nvars, D, device=xb.device).detach()
            # Combine kept and masked patches
            x_ = torch.cat([x_kept, x_removed], dim=0)
            
            # Restore original patch ordering
            x_list[i] = torch.gather(x_, dim=0, index=ids_restore.unsqueeze(-1).repeat(1, 1, D))
            
        x_masked = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
    else:
        bs, L, nvars, D = x.shape
        len_keep = int(L * (1 - mask_ratio))
        
        # Generate noise and sort for masking
        noise = torch.rand(bs, L, nvars, device=xb.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        
        # Keep the first subset
        ids_keep = ids_shuffle[:, :len_keep]
        x_kept = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, 1, D))
        
        # Create and combine with zeros for masked positions
        x_removed = torch.zeros(bs, L - len_keep, nvars, D, device=xb.device).detach()
        x_ = torch.cat([x_kept, x_removed], dim=1)
        
        # Restore original order
        x_masked = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, 1, D))
    
    return x_masked

In [ ]:
#| notest
x = torch.randn(50,16,7,50)
mask_ratio = 0.4

x_new = mask_patches_simple(x,mask_ratio=mask_ratio)
x_new.shape, x.shape

(torch.Size([50, 16, 7, 50]), torch.Size([50, 16, 7, 50]))

In [ ]:
#| notest
patch_len = 750
d_model = 12
n_vars = 7
max_len = 100
bs = 3
# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (bs,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(length, n_vars, patch_len) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

x = mask_patches_simple(x_nested, mask_ratio=0.5)
x[0].shape, x[1].shape, x_nested[0].shape, x_nested[1].shape

(torch.Size([70, 7, 750]),
 torch.Size([52, 7, 750]),
 torch.Size([70, 7, 750]),
 torch.Size([52, 7, 750]))

## Value Augmentations

In [ ]:
#| export
def remove_values(x, mask_ratio):
    mask = torch.rand(x.shape, device=x.device) > mask_ratio # maskout_ratio are False, note that this is per channel masking
    x = x*mask
    return x

def jitter_augmentation(x, mask_ratio=0.05, jitter_ratio=0.05, p=1):
    prob = torch.rand(1) < p
    if not prob:
        return x
    if x.is_nested:
        out_tensors = []
        for x_i in x:
            max_amplitude = x_i.abs().max()  # Use absolute max for amplitude scale
            zero_mask = torch.rand(x_i.shape, device=x.device) > mask_ratio
            jitter_mask = torch.rand(x_i.shape, device=x.device) > (1-jitter_ratio)
            jitter_values = torch.randn(x_i.shape, device=x.device) * max_amplitude * 0.1  # Normal distribution centered at 0 times 10% of max amplitude
            
            out_tensors.append((x_i*zero_mask + jitter_mask*jitter_values).transpose(0,1))
        x = torch.nested.as_nested_tensor(out_tensors, layout=torch.jagged)
        x = x.transpose(1,2)
    else:
        x = x.clone()  
        max_amplitude = x.abs().max()  # Use absolute max for amplitude scale

        zero_mask = torch.rand(x.shape, device=x.device) > mask_ratio
        jitter_mask = torch.rand(x.shape, device=x.device) > (1-jitter_ratio)
        jitter_values = torch.randn(x.shape, device=x.device) * max_amplitude * 0.1  # Normal distribution centered at 0 times 10% of max amplitude
        
        x = x*zero_mask + jitter_mask*jitter_values
    return x

In [ ]:
#| notest
patch_len = 750
d_model = 12
n_vars = 7
max_len = 100
bs = 3
# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (bs,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, max_len) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

x = jitter_augmentation(x_nested, mask_ratio=0.9, jitter_ratio=0.05)
x.shape

torch.Size([3, 7, j8])

In [ ]:
#| notest
## note that the random number generator advances state...
torch.manual_seed(42)
x = torch.randn(4,7,1000)

torch.manual_seed(42)
x_new, n_masks = jitter_augmentation(x)
n_masks /(4* 7*1000)

torch.manual_seed(42)
x_new2, n_masks2 = jitter_augmentation(x)
torch.equal(x_new, x_new2)

ValueError: too many values to unpack (expected 2)

## Shuffle Augmentations

In [ ]:
#| export
def shuffle_dim(x, dim=1, p=0.5):
    """
    shuffles a dimension randomly along dim
    x: [bs x n channels x n patches x patch len]
    """
    if torch.rand(1, device=x.device) > (1-p):
        idx = torch.randperm(x.size(dim), device=x.device)
        return torch.index_select(x, dim, idx)
    else:
        return x


In [ ]:
#| export
def reverse_sequence(x, seq_dim=(-1,), p=0.5):
    if torch.rand(1, device=x.device) > (1-p):
        return torch.flip(x, dims=seq_dim)
    else:
        return x

In [ ]:
#| notest
x = torch.randn(4,1,5,5).to('cuda')

torch.equal(shuffle_dim(x), x)

False

In [ ]:
#| export
def channel_masking(x, dim=1, p=0.5, specific_channels=None):
    """
    Masks up to n channels - 1 randomly of x or specific channels if provided
    """
    if specific_channels is not None:
        specific_channels = [specific_channels] if isinstance(specific_channels, int) else specific_channels
        idx = torch.tensor(specific_channels, device=x.device)
        if torch.rand(1, device=x.device) > (1-p):
            if x.is_nested:
                x_list = list(x.unbind())
                for i, x_i in enumerate(x_list):
                    mask = torch.ones_like(x_i, device=x.device) # mask of ones
                    mask[idx] = 0 # set the mask to 0 for the channels to mask
                    x_list[i] = (x_i * mask).transpose(0,1) # apply the mask
                x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
                x = x.transpose(1,2)
            else:
                mask = torch.ones_like(x, device=x.device) # mask of ones
                mask[:,idx] = 0 # set the mask to 0 for the channels to mask
                x = x * mask # apply the mask
    else:
        if torch.rand(1, device=x.device) > (1-p):
            if x.is_nested:
                x_list = list(x.unbind())
                for i, x_i in enumerate(x_list):
                    n_channels_to_mask = torch.randint(1, x_i.size(dim-1), (1,), device=x.device)
                    mask = torch.ones_like(x_i, device=x.device) # mask of ones
                    idx = torch.randperm(x_i.size(dim-1), device=x.device)[:n_channels_to_mask] # get n_channels_to_mask random indices
                    mask[:, idx] = 0 # set the mask to 0 for the channels to mask
                    x_list[i] = (x_i * mask).transpose(0,1) # apply the mask
                x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
                x = x.transpose(1,2)
            else:
                n_channels_to_mask = torch.randint(1, x.size(dim), (1,), device=x.device)
                mask = torch.ones_like(x, device=x.device) # mask of ones
                idx = torch.randperm(x.size(dim), device=x.device)[:n_channels_to_mask] # get n_channels_to_mask random indices
                mask[:,idx] = 0 # set the mask to 0 for the channels to mask
                x = x * mask # apply the mask
    return x

In [ ]:
#| notest
x = torch.randn(4,7,5,5)

x_new = channel_masking(x, dim=1, p=0.5)

patch_len = 750
d_model = 12
n_vars = 7
max_len = 100
bs = 3
# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (bs,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

x = channel_masking(x, dim=1, p=1.5, specific_channels=[0,1])

In [ ]:
#| export
def random_crop(x, c_in, min_len, p=0.5):
        """
        Args:
            x: Either a regular tensor [n_channels x seq_len] 
               or a nested tensor with variable lengths
        Returns:
            Cropped version of input with random length
        """
        prob = torch.rand(1) < p
        if not prob:
             return x
        if x.is_nested:
            # Handle batch of variable length sequences
            cropped_tensors = []
            for x_i in x:
                seq_len = x_i.size(-1)
                if seq_len <= min_len:
                    cropped_tensors.append(x_i)
                    continue
                crop_length = torch.randint(min_len, seq_len, size=(1,)).item()
                crop = RandomCrop(size=(c_in, crop_length)) # size is desired output size
                cropped_tensors.append(crop(x_i).transpose(0,1))
            x = torch.nested.as_nested_tensor(cropped_tensors, layout=torch.jagged)
            x = x.transpose(1,2)
            return x
        else:
            # Handle single sequence
            seq_len = x.size(-1)
            if seq_len <= min_len:
                return x
            crop_length = torch.randint(min_len, seq_len, size=(1,)).item()
            crop = RandomCrop(size=(c_in, crop_length)) # size is desired output size
            return crop(x)


In [ ]:
#| notest
x = torch.randn(1, 7, 100)
print(x.shape)
y = random_crop(x_nested, 7, 10)
print(y.shape)
[y_i.shape for y_i in y]

torch.Size([1, 7, 100])
torch.Size([3, 7, j1])


[torch.Size([7, 60]), torch.Size([7, 58]), torch.Size([7, 83])]

## Transforms

In [ ]:
#| export
class IntraClassCutMix1d(object):
    """
    Intra-class CutMix for 1D data (e.g., time-series). 

    This is a callback that can be used to apply CutMix to the training data.
    It is used to mix segments within the same class.
    """
    def __init__(self, 
                 mix_prob=0.5, # probability of applying cutmix
                 return_y_every_sec=30, # length of segment to mix, if one value of y corresponds to 30 seconds of signal data, this should be set to 30. 
                 frequency=125, # frequency of the data
                 ignore_index=-100,
                 ):
        super().__init__()
        self.mix_prob = mix_prob # probability of applying cutmix
        self.return_y_every_sec = return_y_every_sec # length of segment to mix
        self.frequency = frequency # frequency of the data
        self.ignore_index = ignore_index
    def __call__(self, batch):
        x, y = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]

        if torch.rand(1) < self.mix_prob:
            bs, c_in, _ = x.shape
            was_nested = x.is_nested
            if x.is_nested:
                max_len = max(x.size(-1) for x in x.unbind())
                og_x_seq_len = [i.size(-1) for i in x.unbind()]
                og_y_seq_len = [i.size(-1) for i in y.unbind()]
                x = x.to_padded_tensor(padding=0, output_size=(bs, c_in, max_len))
                y = y.to_padded_tensor(padding=self.ignore_index, output_size=(bs, max_len//(self.return_y_every_sec*self.frequency))).squeeze(-1)
            # Number of segments
            n_segments = y.shape[-1]

            for seg in range(n_segments):
                start = seg
                end = seg + 1

                start_x = start*self.return_y_every_sec*self.frequency
                end_x = end*self.return_y_every_sec*self.frequency
                # Get the classes for this segment across all batch items
                seg_classes = y[:, start].cpu()  # Assuming the class is consistent within a segment

                for cls in seg_classes.unique():
                    # Find batch items with this class in the current segment
                    batch_indices = (seg_classes == cls).nonzero().squeeze(-1)
                    if len(batch_indices) > 1:  # We need at least 2 samples to mix
                        # Shuffle these indices
                        shuffled_indices = batch_indices[torch.randperm(len(batch_indices))]
                        
                        # Mix the segments
                        x[batch_indices, :, start_x:end_x] = x[shuffled_indices, :, start_x:end_x]
            if was_nested:
                x_list = []
                y_list = []
                for i in range(bs):
                    x_list.append(x[i, :, :og_x_seq_len[i]].transpose(0,1))
                    y_list.append(y[i, :og_y_seq_len[i]])
                x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
                y = torch.nested.as_nested_tensor(y_list, layout=torch.jagged)
                x = x.transpose(1,2)
        return (x,y)

In [ ]:
#| notest
x = torch.randn(4,7,90)
x_c = x.clone()
y = torch.randint(0, 5, size=(4,90//30))
xxt = IntraClassCutMix1d(mix_prob=1, frequency=1, return_y_every_sec=30)
batch = (x,y)
batch = xxt(batch)
torch.equal(x_c, batch[0]) == False

True

In [ ]:
#| notest
n_vars = 4
max_len = 90
bs = 3
frequnecy = 1
# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (bs,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
y_list = [torch.randint(0, 5, size=(length,)) for length in seq_lens]
y_nested = torch.nested.as_nested_tensor(y_list, layout=torch.jagged)

xxt = IntraClassCutMix1d(mix_prob=1, frequency=1, return_y_every_sec=30)
batch = (x_nested, y_nested)
batch = xxt(batch)
batch[0], batch[1]

(NestedTensor(size=(3, 4, j167), offsets=tensor([  0,  51, 135, 187]), contiguous=True),
 NestedTensor(size=(3, j168), offsets=tensor([0, 2, 4, 6]), contiguous=True))

In [ ]:
#| export
class IntraClassCutMixBatch(object):
    """
    Intra-class CutMix for 1D data (e.g., time-series). 

    This is a callback that can be used to apply CutMix to the training data.
    It is used to mix segments within the same class.

    This is different to IntraClassCutMix1d in that it mixes segments of the same class across batches of data, rather than just at the same segment
    """
    def __init__(self, 
                 mix_prob=0.5, # probability of applying cutmix
                 return_y_every_sec=30, # length of segment to mix, if one value of y corresponds to 30 seconds of signal data, this should be set to 30. 
                 frequency=125, # frequency of the data
                 ignore_index=-100,
                 intra_class_only=True # whether to mix only within same class (True) or across all classes (False)
                 ):
        super().__init__()
        self.mix_prob = mix_prob # probability of applying cutmix
        self.frequency = frequency # frequency of the data
        self.segment_length = return_y_every_sec*self.frequency # length of segment to mix
        self.intra_class_only = intra_class_only # whether to mix only within same class (True) or across all classes (False)
        self.ignore_index = ignore_index
        
    def __call__(self, batch):
        if torch.rand(1) < self.mix_prob:
            x, y = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
            bs, c_in, _ = x.shape
            was_nested = x.is_nested
            if x.is_nested:
                max_len = max(x.size(-1) for x in x.unbind())
                og_x_seq_len = [i.size(-1) for i in x.unbind()]
                og_y_seq_len = [i.size(-1) for i in y.unbind()]
                x = x.to_padded_tensor(padding=0, output_size=(bs, c_in, max_len))
                y = y.to_padded_tensor(padding=self.ignore_index, output_size=(bs, max_len//self.segment_length)).squeeze(-1)
            seq_len = x.shape[-1]

            # create patches of length segment_length, [bs x n_patches x n_vars x patch_len]
            x = create_patch(x, patch_len=self.segment_length, stride=self.segment_length, constant_pad=False, constant_pad_value=0)
            if self.intra_class_only:
                unique_classes = y.unique()
                for cls in unique_classes:
                    class_indices = (y == cls).nonzero()
                    if len(class_indices) > 1:
                        # shuffle the indices
                        shuffled_indices = class_indices[torch.randperm(len(class_indices))]
                        # Extract the batch and patch indices
                        batch_indices = class_indices[:, 0]
                        patch_indices = class_indices[:, 1]
                        # mix the segments
                        x[batch_indices, patch_indices] = x[shuffled_indices[:, 0], shuffled_indices[:, 1]]
            else:
                # Cross-class mixing logic
                all_indices = torch.nonzero(torch.ones_like(y))  # Get all valid indices
                if len(all_indices) > 1:
                    shuffled_indices = all_indices[torch.randperm(len(all_indices))]
                    batch_indices = all_indices[:, 0]
                    patch_indices = all_indices[:, 1]
                    x[batch_indices, patch_indices] = x[shuffled_indices[:, 0], shuffled_indices[:, 1]]
                    y[batch_indices, patch_indices] = y[shuffled_indices[:, 0], shuffled_indices[:, 1]]
            # unpatch the data
            x = unpatch(x, seq_len)
            if was_nested:
                x_list = []
                y_list = []
                for i in range(bs):
                    x_list.append(x[i, :, :og_x_seq_len[i]].transpose(0,1))
                    y_list.append(y[i, :og_y_seq_len[i]])
                x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
                y = torch.nested.as_nested_tensor(y_list, layout=torch.jagged)
                x = x.transpose(1,2)
        return (x,y)

In [ ]:
#| notest
x = torch.randn(4,7,90)
x_c = x.clone()
y = torch.randint(0, 5, size=(4,90//30))
xxt = IntraClassCutMixBatch(mix_prob=1, frequency=1, return_y_every_sec=30)
batch = [x,y]
batch = xxt(batch)
torch.equal(x_c, batch[0]) == False

In [ ]:
#| notest

n_vars = 4
max_len = 90
bs = 3
frequnecy = 1
# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (bs,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
y_list = [torch.randint(0, 5, size=(length,)) for length in seq_lens]
y_nested = torch.nested.as_nested_tensor(y_list, layout=torch.jagged)

xxt = IntraClassCutMixBatch(mix_prob=1, frequency=1, return_y_every_sec=30)
batch = [x_nested, y_nested]
batch = xxt(batch)
batch[0], batch[1]

In [ ]:
#| export
class MixupCallbackSequence(object):
    """
    Mixup for 1D data (e.g., time-series).

    This callback applies Mixup to the training data, blending both the input data and the labels.

    See tsai implementation here: https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/data/mixed_augmentation.py#L43

    Note that this creates non-integer labels/soft labels. Loss functions should be able to handle this.
    """
    def __init__(self, 
                 num_classes,
                 return_y_every_sec=30,
                 frequency=None,
                 mixup_alpha=0.4, # alpha parameter for the beta distribution
                 ignore_index=-100 # ignore index
                 ):
        super().__init__()
        self.distrib = Beta(mixup_alpha, mixup_alpha)
        self.mixup_alpha = mixup_alpha
        self.ignore_index = ignore_index
        self.num_classes = num_classes
        self.return_y_every_sec = return_y_every_sec
        self.frequency = frequency

    def __call__(self, batch):
        x, y = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
        bs, c_in, _ = x.shape
        was_nested = x.is_nested
        if x.is_nested:
            max_len = max(x.size(-1) for x in x.unbind())
            og_x_seq_len = [i.size(-1) for i in x.unbind()]
            og_y_seq_len = [i.size(-1) for i in y.unbind()]
            x = x.to_padded_tensor(padding=0, output_size=(bs, c_in, max_len))
            y = y.to_padded_tensor(padding=self.ignore_index, output_size=(bs, max_len//(self.return_y_every_sec*self.frequency))).squeeze(-1)

        # Sample lambda from a beta distribution
        lam = self.distrib.sample((x.size(0), )).to(x.device) # [bs]
        # our mixing coefficient is always ≥ 0.5
        #lam = torch.max(lam, 1 - lam)
        lam = lam.view(-1, 1, 1)  # for input shape [bs, channels, seq_len]

        # Shuffle the batch
        indices = torch.randperm(x.size(0), device=x.device)
    
        x_shuffled = x[indices]
        y_shuffled = y[indices]

        # create ignore masks
        ignore_mask = (y == self.ignore_index)
        ignore_mask_shuffled = (y_shuffled == self.ignore_index)
        combined_ignore_mask = torch.logical_or(ignore_mask, ignore_mask_shuffled)

        y_clean = torch.where(ignore_mask, torch.zeros_like(y), y)
        y_shuffled_clean = torch.where(ignore_mask_shuffled, torch.zeros_like(y_shuffled), y_shuffled)
        # Create one-hot encodings
        y_onehot = F.one_hot(y_clean.long(), num_classes=self.num_classes).float()
        y_shuffled_onehot = F.one_hot(y_shuffled_clean.long(), num_classes=self.num_classes).float()
        # Zero out the one-hot vectors for ignored indices
        y_onehot = torch.where(ignore_mask.unsqueeze(-1), torch.zeros_like(y_onehot), y_onehot)
        y_shuffled_onehot = torch.where(ignore_mask_shuffled.unsqueeze(-1), torch.zeros_like(y_shuffled_onehot), y_shuffled_onehot)

        # Mixup the inputs and labels
        x_mixed = torch.lerp(x, x_shuffled, lam) # x = x_shuffled + lam * (x - x_shuffled)
        y_mixed = torch.lerp(y_onehot, y_shuffled_onehot, lam) # y = y_shuffled + lam * (y - y_shuffled)

        # finally assign all probabilities with a -100 mixing to 0
        y_mixed = torch.where(combined_ignore_mask.unsqueeze(-1), torch.zeros_like(y_mixed), y_mixed)
        # # add back in the ignore index where it was mixed with other labels
        # combined_ignore_mask = torch.logical_or(ignore_mask, ignore_mask_shuffled)
        # if combined_ignore_mask.any():
        #     # y_mixed = torch.where(combined_ignore_mask, 
        #     #                         torch.tensor(self.ignore_index, dtype=y_mixed.dtype, device=y_mixed.device), 
        #     #                     y_mixed)
        #     y_mixed = torch.where(
        #         combined_ignore_mask.unsqueeze(-1),
        #         torch.zeros_like(y_mixed),
        #         y_mixed
        #     )
        y_mixed = y_mixed.permute(0,2,1) # [bs x n_classes x n_patches]
        if was_nested:
            x_list = []
            y_list = []
            for i in range(bs):
                x_list.append(x_mixed[i, :, :og_x_seq_len[i]].transpose(0,1))
                y_list.append(y_mixed[i, :, :og_y_seq_len[i]].transpose(0,1))
            x_mixed = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
            y_mixed = torch.nested.as_nested_tensor(y_list, layout=torch.jagged)
            x_mixed = x_mixed.transpose(1,2)
            y_mixed = y_mixed.transpose(1,2)
        return (x_mixed, y_mixed)

In [ ]:
#| notest
x = torch.randn(4,7,90)
x_c = x.clone()
y_og = torch.randint(0, 2, size=(4,90//30))
y_og[1,2] = -100
y_og[2,1] = -100
y_c = y_og.clone()
xxt = MixupCallbackSequence(num_classes=2, mixup_alpha=0.4)
batch = (x,y_og)
batch = xxt(batch)
torch.equal(x_c, batch[0]) == False, torch.equal(y_c, batch[1]) == False

(True, True)

In [ ]:
#| export
class MixupCallbackClassification(object):
    """
    Mixup for 1D data (e.g., time-series).

    This callback applies Mixup to the training data, blending both the input data and the labels.

    See tsai implementation here: https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/data/mixed_augmentation.py#L43

    Note that this creates non-integer labels/soft labels. Loss functions should be able to handle this.
    """
    def __init__(self, 
                 num_classes,
                 mixup_alpha=0.4, # alpha parameter for the beta distribution
                 ignore_index=-100 # ignore index
                 ):
        super().__init__()
        self.distrib = Beta(mixup_alpha, mixup_alpha)
        self.mixup_alpha = mixup_alpha
        self.ignore_index = ignore_index
        self.num_classes = num_classes

    def __call__(self, batch):
        x, y = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
        bs, c_in, _ = x.shape
        was_nested = x.is_nested
        if x.is_nested:
            max_len = max(x.size(-1) for x in x.unbind())
            og_x_seq_len = [i.size(-1) for i in x.unbind()]
            x = x.to_padded_tensor(padding=0, output_size=(bs, c_in, max_len))

        # Sample lambda from a beta distribution
        lam = self.distrib.sample((x.size(0), )).to(x.device) # [bs]
        # our mixing coefficient is always ≥ 0.5
        lam = torch.max(lam, 1 - lam)
        lam_x = lam.view(-1, 1, 1)  # for input shape [bs, channels, seq_len]
        lam_y = lam.view(-1, 1)

        # Shuffle the batch
        indices = torch.randperm(x.size(0), device=x.device)
    
        x_shuffled = x[indices]
        y_shuffled = y[indices]

        # create ignore masks
        ignore_mask = (y == self.ignore_index)
        ignore_mask_shuffled = (y_shuffled == self.ignore_index)
        combined_ignore_mask = torch.logical_or(ignore_mask, ignore_mask_shuffled)

        y_clean = torch.where(ignore_mask, torch.zeros_like(y), y).long()
        y_shuffled_clean = torch.where(ignore_mask_shuffled, torch.zeros_like(y_shuffled), y_shuffled).long()
        # Create one-hot encodings
        if y.ndim == 1:
            y_onehot = F.one_hot(y_clean, num_classes=self.num_classes).float()
            y_shuffled_onehot = F.one_hot(y_shuffled_clean, num_classes=self.num_classes).float()
            ignore_mask = ignore_mask.unsqueeze(-1)
            ignore_mask_shuffled = ignore_mask_shuffled.unsqueeze(-1)
            combined_ignore_mask = combined_ignore_mask.unsqueeze(-1)
        else:
            y_onehot = y_clean.float()
            y_shuffled_onehot = y_shuffled_clean.float()
        # Zero out the one-hot vectors for ignored indices
        y_onehot = torch.where(ignore_mask, torch.zeros_like(y_onehot), y_onehot)
        y_shuffled_onehot = torch.where(ignore_mask_shuffled, torch.zeros_like(y_shuffled_onehot), y_shuffled_onehot)
        # Mixup the inputs and labels
        x_mixed = torch.lerp(x, x_shuffled, lam_x) # x = x_shuffled + lam * (x - x_shuffled)
        y_mixed = torch.lerp(y_onehot, y_shuffled_onehot, lam_y) # y = y_shuffled + lam * (y - y_shuffled)

        # finally assign all probabilities with a -100 mixing to 0
        y_mixed = torch.where(combined_ignore_mask, torch.zeros_like(y_mixed), y_mixed)
        # # add back in the ignore index where it was mixed with other labels
        if was_nested:
            x_list = []
            for i in range(bs):
                x_list.append(x_mixed[i, :, :og_x_seq_len[i]].transpose(0,1))
            x_mixed = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
            x_mixed = x_mixed.transpose(1,2)
        return (x_mixed, y_mixed)

In [ ]:
#| notest
x = torch.randn(4,7,90)
x_c = x.clone()
y_og = torch.randint(0, 2, size=(4,2))
print(y_og, y_og.shape)
y_c = y_og.clone()
xxt = MixupCallbackClassification(num_classes=2, mixup_alpha=0.4)
batch = (x,y_og)
batch = xxt(batch)
torch.equal(x_c, batch[0]) == False, torch.equal(y_c, batch[1]) == False

tensor([[0, 1],
        [0, 0],
        [0, 1],
        [0, 1]]) torch.Size([4, 2])
tensor([[0., 1.],
        [0., 0.],
        [0., 1.],
        [0., 1.]]) tensor([[0., 1.],
        [0., 0.],
        [0., 1.],
        [0., 1.]])


(True, False)

In [ ]:
#| notest
n_vars = 4
max_len = 90
bs = 3
frequnecy = 1
# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (bs,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
y_list = [torch.randint(0, 5, size=(length,)) for length in seq_lens]
y_nested = torch.nested.as_nested_tensor(y_list, layout=torch.jagged)

batch = [x_nested, y_nested]

xxt = MixupCallbackSequence(num_classes=5, mixup_alpha=0.4, return_y_every_sec=30, frequency=frequnecy)
batch = [x_nested, y_nested]
batch = xxt(batch)
batch[0], batch[1]

(NestedTensor(size=(3, 4, j173), offsets=tensor([  0,  63, 133, 204]), contiguous=True),
 NestedTensor(size=(3, 5, j174), offsets=tensor([0, 2, 4, 6]), contiguous=True))

In [ ]:
#| export
class MixupTimeDemos(object):
    """
    Mixup for 1D data and time to event data + demographics

    This callback applies Mixup to the training data, blending both the input data and the labels.

    See tsai implementation here: https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/data/mixed_augmentation.py#L43

    Note that this creates non-integer labels/soft labels. Loss functions should be able to handle this.
    """
    def __init__(self, 
                 mixup_alpha=0.4, # alpha parameter for the beta distribution
                 mix_time=False, # whether to mix time to event
                 mix_demographics=False, # whether to mix demographics
                 ):
        super().__init__()
        self.distrib = Beta(mixup_alpha, mixup_alpha)
        self.mixup_alpha = mixup_alpha
        self.mix_time = mix_time
        self.mix_demographics = mix_demographics

    def __call__(self, batch):
        x, y, time, demographics = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
        bs, c_in, _ = x.shape
        was_nested = x.is_nested
        if x.is_nested:
            max_len = max(x.size(-1) for x in x.unbind())
            og_x_seq_len = [i.size(-1) for i in x.unbind()]
            x = x.to_padded_tensor(padding=0, output_size=(bs, c_in, max_len))

        # Sample lambda from a beta distribution
        lam = self.distrib.sample((x.size(0), )).to(x.device) # [bs]
        # our mixing coefficient is always ≥ 0.5
        lam = torch.max(lam, 1 - lam)
        lam_x = lam.view(-1, 1, 1)  # for input shape [bs, channels, seq_len]
        lam_y = lam.view(-1, 1)

        # Shuffle the batch
        indices = torch.randperm(x.size(0), device=x.device)
    
        x_shuffled = x[indices]
        y_shuffled = y[indices]
        time_shuffled = time[indices]
        demographics_shuffled = demographics[indices]

        # Create one-hot encodings
        y_onehot = F.one_hot(y, num_classes=2).float()
        y_shuffled_onehot = F.one_hot(y_shuffled, num_classes=2).float()
        # Mixup the inputs and labels
        x_mixed = torch.lerp(x, x_shuffled, lam_x) # x = (1 - lam) * x + lam * x_shuffled
        y_mixed = torch.lerp(y_onehot, y_shuffled_onehot, lam_y) 
        if self.mix_time:
            time_mixed = torch.lerp(time, time_shuffled, lam)
        else:
            time_mixed = time_shuffled
        if self.mix_demographics:
            demographics_mixed = torch.lerp(demographics.float(), demographics_shuffled.float(), lam_y)
        else:
            # with lambda >= 0.5, the dominating sample is x_shuffled
            demographics_mixed = demographics_shuffled

        if was_nested:
            x_list = []
            for i in range(bs):
                x_list.append(x_mixed[i, :, :og_x_seq_len[i]].transpose(0,1))
            x_mixed = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
            x_mixed = x_mixed.transpose(1,2)
        return (x_mixed, y_mixed, time_mixed, demographics_mixed)

In [ ]:
#| notest
x = torch.randn(4,7,90)
x_c = x.clone()
y_og = torch.randint(0, 2, (4,))
time_idx = torch.randint(0, 18, (4,)).float()
age_demographics = torch.randint(20, 80, (4,)).float()
gender_demographics = torch.randint(0, 2, (4,)).float()
demographics = torch.stack([age_demographics, gender_demographics], dim=1)
print(demographics, demographics.shape)
print(y_og, y_og.shape)
y_c = y_og.clone()
xxt = MixupTimeDemos(mixup_alpha=0.4)
batch = (x,y_og, time_idx, demographics)
batch = xxt(batch)
torch.equal(x_c, batch[0]) == False, torch.equal(y_c, batch[1]) == False

tensor([[45.,  1.],
        [62.,  1.],
        [32.,  0.],
        [44.,  1.]]) torch.Size([4, 2])
tensor([1, 0, 0, 1]) torch.Size([4])


(True, True)

In [ ]:
#| export
class MixupTimeDemosNoX(object):
    """
    Mixup for 1D data and time to event data + demographics

    This callback applies Mixup to the training data, blending both the input data and the labels.

    See tsai implementation here: https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/data/mixed_augmentation.py#L43

    Note that this creates non-integer labels/soft labels. Loss functions should be able to handle this.
    """
    def __init__(self, 
                 mixup_alpha=0.4, # alpha parameter for the beta distribution
                 mix_time=False, # whether to mix time to event
                 ):
        super().__init__()
        self.distrib = Beta(mixup_alpha, mixup_alpha)
        self.mixup_alpha = mixup_alpha
        self.mix_time = mix_time

    def __call__(self, batch):
        *x, y, time, demographics = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
        # Sample lambda from a beta distribution
        lam = self.distrib.sample((y.size(0), )).to(y.device) # [bs]
        # our mixing coefficient is always ≥ 0.5
        lam = torch.max(lam, 1 - lam)
        lam_y = lam.view(-1, 1)

        # Shuffle the batch
        indices = torch.randperm(y.size(0), device=y.device)
    
        y_shuffled = y[indices]
        time_shuffled = time[indices]
        demographics_shuffled = demographics[indices]

        # Create one-hot encodings
        y_onehot = F.one_hot(y, num_classes=2).float()
        y_shuffled_onehot = F.one_hot(y_shuffled, num_classes=2).float()
        # Mixup the inputs and labels
        y_mixed = torch.lerp(y_onehot, y_shuffled_onehot, lam_y) # y = y_shuffled + lam * (y - y_shuffled)
        if self.mix_time:
            time_mixed = torch.lerp(time, time_shuffled, lam)
        else:
            time_mixed = time_shuffled
        # demographics always mixed bc demographics only model
        demographics_mixed = torch.lerp(demographics.float(), demographics_shuffled.float(), lam_y)

        if len(x) > 0:
            x = x[0]
            return (x, y_mixed, time_mixed, demographics_mixed)
        else:
            return (y_mixed, time_mixed, demographics_mixed)

In [ ]:
#| export
class TransformsCallback(object):
    """
    Applies a series of transforms to the input data, on train_batch_start.
    """
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, batch):
        if len(batch) == 3:
            x, y, time = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
        else:
            x, y = batch
        for transform in self.transforms:
            x = transform(x)
        if len(batch) == 3:
            return (x,y,time)
        else:
            return (x,y)

In [ ]:
#| export
class MixupPretrainCallback(object):
    """
    Mixup for 1D data (e.g., time-series).

    This callback applies Mixup to the training data, blending both the input data

    See tsai implementation here: https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/data/mixed_augmentation.py#L43
    """
    def __init__(self, 
                 mixup_alpha=0.4, # alpha parameter for the beta distribution
                 ):
        super().__init__()
        self.distrib = Beta(mixup_alpha, mixup_alpha)
        self.mixup_alpha = mixup_alpha

    def __call__(self, batch):
        if len(batch) == 3:
            x, y, time = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
        else:
            x, y = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
        bs, c_in, _ = x.shape
        was_nested = x.is_nested
        if x.is_nested:
            max_len = max(x.size(-1) for x in x.unbind())
            og_x_seq_len = [i.size(-1) for i in x.unbind()]
            x = x.to_padded_tensor(padding=0, output_size=(bs, c_in, max_len))

        lam = self.distrib.sample((x.size(0), )).to(x.device) # [bs]
        # our mixing coefficient is always ≥ 0.5
        lam = torch.max(lam, 1 - lam)
        lam = lam.view(-1, 1, 1)  # for input shape [bs, channels, seq_len]

        # Shuffle the batch
        indices = torch.randperm(x.size(0), device=x.device)
    
        x_shuffled = x[indices]

        # Mixup the inputs and labels
        x_mixed = torch.lerp(x, x_shuffled, lam) # x = x_shuffled + lam * (x - x_shuffled)
        if was_nested:
            x_list = []
            for i in range(bs):
                x_list.append(x_mixed[i, :, :og_x_seq_len[i]].transpose(0,1))
            x_mixed = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
            x_mixed = x_mixed.transpose(1,2)
        if len(batch) == 3:
            return (x_mixed, y, time)
        else:
            return (x_mixed, y)

In [ ]:
#| export
class VariableChannelInput(object):
    """
    Randomly adds 0 channels at correct index to the input data to match the number of channels expected by the self supervised model.
    """
    def __init__(self, indexes_to_add_channels, n_channels_expected, channel_dim=1):
        super().__init__()
        if not isinstance(indexes_to_add_channels, list) and not isinstance(indexes_to_add_channels, tuple):
            raise ValueError("indexes_to_add_channels must be a list or tuple")
        if not all(isinstance(i, int) for i in indexes_to_add_channels):
            raise ValueError("indexes_to_add_channels must be a list or tuple of integers")
        self.indexes_to_add_channels = indexes_to_add_channels
        self.n_channels_expected = n_channels_expected
        self.channel_dim = channel_dim
    
    def _modify_batch(self, x):
        assert x.shape[self.channel_dim] + len(self.indexes_to_add_channels) == self.n_channels_expected, f"The number of channels expected ({self.n_channels_expected}) by the model must match the number of channels in the input data ({x.shape[self.channel_dim]}) plus the number of channels to add ({len(self.indexes_to_add_channels)})"
        # append zeros to the channels at the end of the input data

        if x.is_nested:
            x_list = list(x.unbind())
            for i, x_i in enumerate(x_list):
                x_list[i] = torch.cat([x_i, torch.zeros(len(self.indexes_to_add_channels), x_i.shape[1], device=x.device)], dim=self.channel_dim-1)
            x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        else:
            x = torch.cat([x, torch.zeros(x.shape[0], len(self.indexes_to_add_channels), x.shape[2], device=x.device)], dim=self.channel_dim)
        # reindex to match the correct channel order
        new_indices = list(range(self.n_channels_expected))
        # find the added 0 indices and their new indices
        index_to_add_dict = dict(zip(new_indices[-len(self.indexes_to_add_channels):], self.indexes_to_add_channels))
        for old_idx, new_idx in index_to_add_dict.items():
            new_indices.insert(new_idx, new_indices.pop(old_idx))
        if x.is_nested:
            x_list = list(x.unbind())
            for i, x_i in enumerate(x_list):
                x_list[i] = torch.index_select(x_i, dim=self.channel_dim-1, index=torch.tensor(new_indices, device=x.device))
            x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        else:
            x = torch.index_select(x, dim=self.channel_dim, index=torch.tensor(new_indices, device=x.device))
        return x
    
    def reverse(self, x):
        # reverse the channel addition
        if x.is_nested:
            x_list = list(x.unbind())
            for i, x_i in enumerate(x_list):
                x_list[i] = torch.index_select(x_i, dim=self.channel_dim-1, index=torch.tensor([i for i in range(x_i.shape[self.channel_dim-1]) if i not in self.indexes_to_add_channels], device=x.device))
            x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        else:
            x = torch.index_select(x, dim=self.channel_dim, index=torch.tensor([i for i in range(x.shape[self.channel_dim]) if i not in self.indexes_to_add_channels], device=x.device))
        return x
        
    def __call__(self, x):
        return self._modify_batch(x)

In [ ]:
callback = VariableChannelInput(
    indexes_to_add_channels=[0],  # Add channel at index 0
    n_channels_expected=5,        # Total channels expected
    channel_dim=1                 # Channel dimension
)

x1 = torch.randn(4,4,90)
x2 = callback(x1)
x1.shape, x2.shape

(torch.Size([4, 4, 90]), torch.Size([4, 5, 90]))

In [ ]:
#| notest
n_vars = 4
max_len = 100
bs = 3
# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (bs,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

callback = VariableChannelInput(
    indexes_to_add_channels=[0],  # Add channel at index 0
    n_channels_expected=5,        # Total channels expected
    channel_dim=1                 # Channel dimension
)

x_nested2 = callback(x_nested)
x_nested.shape, x_nested2.shape

(torch.Size([3, 4, j1]), torch.Size([3, 5, j3]))

In [ ]:
#| notest
x_reversed = callback.reverse(x_nested2)
torch.equal(x_nested.to_padded_tensor(0), x_reversed.to_padded_tensor(0))

True

In [ ]:
#| notest

# Create a simple dataset
class SimpleDataset(Dataset):
    def __init__(self, num_samples=100, seq_length=90, num_channels=4):
        self.data = torch.randn(num_samples, num_channels, seq_length)  # [samples, channels, seq_len]
        self.labels = torch.randint(0, 5, (num_samples,))  # Random labels 0-4
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Create dataset and dataloader
dataset = SimpleDataset()
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Create callback instance
# Let's say we want to add a zero channel at index 0, 
# and our model expects 5 channels total (4 original + 1 added)
callback = VariableChannelInput(
    indexes_to_add_channels=[0],  # Add channel at index 0
    n_channels_expected=5,        # Total channels expected
    channel_dim=1                 # Channel dimension
)

# Test the callback
for batch in dataloader:
    print("Original batch shape:", batch[0].shape)  # Should be [16, 4, 90]
    
    # Simulate what happens in Lightning
    callback.on_train_batch_start(None, None, batch, 0)
    print("Modified batch shape:", batch[0].shape)  # Should be [16, 5, 90]
    
    # Check that the first channel is zeros
    print("First channel is all zeros:", torch.all(batch[0][:, 0] == 0))
    
    break  # Just test one batch

Original batch shape: torch.Size([16, 4, 90])
Modified batch shape: torch.Size([16, 5, 90])
First channel is all zeros: tensor(True)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()